In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import string
import os

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
n_embd = 256
block_size = 256  # Reduced from 512 to handle larger datasets more efficiently
n_head = 8
n_layer = 6  # Reduced from 8 for faster training
dropout = 0.2
batch_size = 64
learning_rate = 3e-4
pre_train_iters = 15000  # Iterations for pre-training
fine_tune_iters = 11000   # Iterations for fine-tuning
eval_interval = 500
eval_iters = 200
max_new_tokens = 400

# Function to read text files
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

# Process text data
def process_text(text, max_chars=None):
    if max_chars and len(text) > max_chars:
        text = text[:max_chars]  # Limit text size if needed
    
    # Filter characters - include more punctuation for better text quality
    allowed_chars = string.ascii_letters + ' ' + '.,!?;:"\'-()\n'
    filtered_text = ''.join([char for char in text if char in allowed_chars])
    return filtered_text

# Create character-level vocabulary from text
def create_vocab(text):
    chars = sorted(list(set(text)))
    char_to_idx = {char: idx for idx, char in enumerate(chars)}
    idx_to_char = {idx: char for idx, char in enumerate(chars)}
    return chars, char_to_idx, idx_to_char

# Text to sequence conversion
def text_to_sequence(text, char_to_idx):
    return [char_to_idx.get(char, 0) for char in text]  # Default to 0 if char not found

# Data loading function for training
def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Loss and accuracy estimation
@torch.no_grad()
def estimate_loss_and_accuracy(model, data, eval_iters, block_size, batch_size):
    model.eval()
    losses = torch.zeros(eval_iters)
    accuracies = torch.zeros(eval_iters)
    
    for k in range(eval_iters):
        X, Y = get_batch(data, block_size, batch_size)
        logits, loss = model(X, Y)
        losses[k] = loss.item()
        
        _, preds = torch.max(logits, dim=-1)
        preds = preds.view(-1)
        Y = Y.view(-1)
        accuracy = (preds == Y).float().mean().item()
        accuracies[k] = accuracy
        
    model.train()
    return {'loss': losses.mean().item(), 'accuracy': accuracies.mean().item()}

# Transformer components
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ Linear layer followed by non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual connections around each component
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate text with optional temperature sampling"""
        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -block_size:] if idx.size(1) > block_size else idx
            logits, _ = self(idx_cond, None)
            # Focus only on the last step's predictions
            logits = logits[:, -1, :] / temperature  # Apply temperature
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Main training and fine-tuning process
def main():
    print("Loading BookCorpus.txt for pre-training...")
    try:
        book_text = read_text_file('/kaggle/input/poems-idk/bookcorpus_clean.txt')
        # Limit to first 5MB for memory constraints if needed
        book_text = process_text(book_text, max_chars=5_000_000)
    except FileNotFoundError:
        print("BookCorpus file not found! Creating a sample text file for demonstration.")
        # Create a sample file if not found
        sample_text = "This is a sample text for demonstration purposes. " * 1000
        with open('bookcorpus.txt', 'w') as f:
            f.write(sample_text)
        book_text = process_text(sample_text)

    print("Creating vocabulary from BookCorpus...")
    chars, char_to_idx, idx_to_char = create_vocab(book_text)
    vocab_size = len(chars)
    print(f"Vocabulary size: {vocab_size}")

    # Convert book text to sequence
    book_seq = text_to_sequence(book_text, char_to_idx)
    book_data = torch.tensor(book_seq, dtype=torch.long, device=device)
    
    print("Initializing model...")
    model = GPTLanguageModel(vocab_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    # Pre-training phase
    print("Starting pre-training on BookCorpus...")
    for iter in range(pre_train_iters):
        if iter % eval_interval == 0:
            metrics = estimate_loss_and_accuracy(model, book_data, eval_iters, block_size, batch_size)
            print(f"Pre-training step {iter}: loss {metrics['loss']:.4f}, accuracy {metrics['accuracy']:.4f}")
            
        xb, yb = get_batch(book_data, block_size, batch_size)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    
    print("Saving pre-trained model checkpoint...")
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char
    }, 'pretrained_model.pth')
    
    # Fine-tuning phase
    print("Loading poems.txt for fine-tuning...")
    try:
        poem_text = read_text_file('/kaggle/input/poems-idk/poems_lowercase.txt')
        poem_text = process_text(poem_text)
    except FileNotFoundError:
        print("Poem file not found! Creating a sample poem file for demonstration.")
        # Create a sample poem file if not found
        sample_poems = """
        Roses are red,
        Violets are blue,
        Sugar is sweet,
        And so are you.
        
        The road not taken,
        Two roads diverged in a yellow wood,
        And sorry I could not travel both,
        And be one traveler, long I stood,
        And looked down one as far as I could.
        """
        with open('poem.txt', 'w') as f:
            f.write(sample_poems)
        poem_text = process_text(sample_poems)
    
    # Convert poem text to sequence using the same vocabulary
    poem_seq = text_to_sequence(poem_text, char_to_idx)
    poem_data = torch.tensor(poem_seq, dtype=torch.long, device=device)
    
    # Fine-tuning optimizer with lower learning rate
    fine_tune_lr = learning_rate / 5  # Lower learning rate for fine-tuning
    optimizer = torch.optim.AdamW(model.parameters(), lr=fine_tune_lr)
    
    print("Starting fine-tuning on poem data...")
    for iter in range(fine_tune_iters):
        if iter % eval_interval == 0:
            metrics = estimate_loss_and_accuracy(model, poem_data, eval_iters, block_size, batch_size)
            print(f"Fine-tuning step {iter}: loss {metrics['loss']:.4f}, accuracy {metrics['accuracy']:.4f}")
            
        xb, yb = get_batch(poem_data, block_size, batch_size)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    
    print("Saving fine-tuned model...")
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char
    }, 'finetuned_poem_model.pth')
    
    # Generate some example poems with different prompts
    prompts = [
        "Love",
        "The moonlight",
        "In the garden",
        "Dreams of",
        "Once upon a time"
    ]
    
    print("\nGenerating sample poems:")
    for prompt in prompts:
        print(f"\n--- Poem starting with '{prompt}' ---")
        initial_indices = [char_to_idx.get(char, 0) for char in prompt]
        context = torch.tensor(initial_indices, dtype=torch.long, device=device).unsqueeze(0)
        generated = model.generate(context, max_new_tokens=200, temperature=0.8)
        generated_text = ''.join([idx_to_char[idx.item()] for idx in generated[0]])
        print(generated_text)

if __name__ == "__main__":
    main()

Using device: cuda
Loading BookCorpus.txt for pre-training...
Creating vocabulary from BookCorpus...
Vocabulary size: 38
Initializing model...
Starting pre-training on BookCorpus...
Pre-training step 0: loss 3.7283, accuracy 0.0052
Pre-training step 500: loss 1.8066, accuracy 0.4489
Pre-training step 1000: loss 1.3158, accuracy 0.5916
Pre-training step 1500: loss 1.1891, accuracy 0.6281
Pre-training step 2000: loss 1.1282, accuracy 0.6463
Pre-training step 2500: loss 1.0870, accuracy 0.6582
Pre-training step 3000: loss 1.0610, accuracy 0.6660
Pre-training step 3500: loss 1.0368, accuracy 0.6732
Pre-training step 4000: loss 1.0230, accuracy 0.6771
Pre-training step 4500: loss 1.0101, accuracy 0.6810
Pre-training step 5000: loss 0.9964, accuracy 0.6852
Pre-training step 5500: loss 0.9867, accuracy 0.6887
Pre-training step 6000: loss 0.9761, accuracy 0.6916
Pre-training step 6500: loss 0.9680, accuracy 0.6941
Pre-training step 7000: loss 0.9585, accuracy 0.6967
Pre-training step 7500: los